# L4 33 — powered Qwen 3B IPD test

Runs the frozen post-pilot design: 40 matched seeds, 20 rounds, and all six controls in IPD. The pre-registered primary endpoint is end cooperation; primary contrasts are trained vs text and trained vs shuffled.

Run cells from top to bottom. Re-running resumes from the Drive checkpoint.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = 'e38b5a51db85dbf351907f00ca35979af9e9fbda'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
JOB_ID = 'faithful-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
PROFILE = 'ipd_confirmatory'
print('Job:', JOB_ID, '| profile:', PROFILE)

In [ ]:
JOB_DIR = pathlib.Path('/content/drive/MyDrive/rival-arena-l4') / JOB_ID
assert (JOB_DIR/'faithful_link.pt').exists(), 'Missing trained link in Drive'
assert (JOB_DIR/'validation_report.json').exists(), 'Missing validation report in Drive'
command = ['python', 'scripts/run_arena.py', '--model', MODEL, '--job-dir', JOB_DIR, '--job-id', JOB_ID, '--profile', PROFILE]
print(' '.join(map(str, command)))
subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected output: `MyDrive/rival-arena-l4/faithful-qwen3b-t4-001/arena_ipd_confirmatory_v3/`. It contains 240 checkpointed matches, so an interrupted runtime can safely resume.